# **Generate Masks**

This notebook processes long `.tif` time series by segmenting each frame individually using a pretrained `Cellpose` model and tracking masks across frames to ensure consistent cell labels over time.

### **Import Libraries**

We begin by importing the necessary packages, including:
- `Cellpose` model API
- `tifffile` for reading/writing `.tif` stacks
- `numpy`, `tqdm`, and `skimage` for image and mask manipulation
- `cct_utils`, which contains our custom tracking logic

In [ ]:
import os
import tifffile
import numpy as np
from tqdm import tqdm
from cellpose import models
from skimage.measure import regionprops
from scipy.spatial.distance import cdist
cct_utils = __import__('0_cct_utils')

### **Define Paths**

Here we define the relevant paths:
- The `.tif` file to process
- The location of the pretrained `Cellpose` model
- The directory for saving masks

In [ ]:
model_name = "cellpose_1746802277.9571602"
model_folder = "ModelAB1"
file_name = 'CON_060525_cluster1_20min_10i_340ms.tif'

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
raw_path = os.path.normpath(os.path.join(parent_dir, "raw_data", model_folder))
mask_path = os.path.normpath(os.path.join(parent_dir, "masks_tracked", model_folder))
model_path = os.path.normpath(os.path.join(parent_dir, "saved_models", model_folder, "cellpose_train", "models", model_name))

os.makedirs(mask_path, exist_ok=True)

### **Tracking Function**

The `track_Y` function:
- Segments each frame individually using the pretrained `Cellpose` model
- Tracks cells across frames using custom tracking logic from `cct_utils`

In [ ]:
def track_Y(X, model, diam=None):
    crop_idx = np.argmax(np.mean(X, axis=(1, 2)) == 0)
    X = X[:crop_idx] if crop_idx > 0 else X
    if len(X) == 0:
        return np.zeros_like(X)
    Y = [model.eval(np.squeeze(i), diameter=diam, channels=[0, 0], flow_threshold=0.8, cellprob_threshold=0.4, do_3D=False)[0]
         for i in np.split(X, X.shape[0])]
    if len(Y) == 0:
        return np.zeros_like(X)
    tracked = cct_utils.get_tracked_masks(masks=np.array(Y))
    return tracked

### **Load Image Stack and Initialize Model**

- The full `.tif` image stack is loaded
- A pretrained `Cellpose` model is initialized for segmentation

In [ ]:
X = tifffile.imread(os.path.join(raw_path, file_name))
model = models.CellposeModel(gpu=True, pretrained_model=model_path)

### **Execute Segmentation and Tracking**

- Processes the entire time-lapse stack sequentially, frame by frame
- Segments each frame and tracks cells across frames
- Generates a final 3D mask stack with consistent cell labels

In [ ]:
print("Processing entire stack frame-by-frame...")
Y = []
for t in tqdm(range(X.shape[0]), desc="Processing frames"):
    frame = X[t:t+1]  # Single frame with shape (1, H, W)
    segmented = track_Y(frame, model, diam=None)
    Y.append(segmented[0])  # Extract frame from output
final_masks = np.array(Y)

### **Save the Final Masks**

- The final mask stack is saved as a `.tif` file in the designated output directory

In [ ]:
output_path = os.path.join(mask_path, file_name)
tifffile.imwrite(output_path, final_masks, imagej=True, metadata={'axes': 'TYX'})
print(f"Final masks saved to: {output_path}")